# 2. Regresión Logística — Predicting Airline Satisfaction

Primer modelo del episodio: un **baseline lineal e interpretable** sobre el que medir todo lo que
venga después.

- **Métrica oficial:** ROC AUC. Se evalúa con `StratifiedKFold(5)` sobre `train`.
- **Nulos:** sólo 292 en `Arrival Delay in Minutes`. Se rellenan con la demora de salida
  (correlación 0,84) y el `SimpleImputer` queda en el pipeline como resguardo.
- **Escalado:** obligatorio en un modelo lineal regularizado. Además vuelve comparables los
  coeficientes entre sí.

El EDA dejó cuatro hipótesis para poner a prueba acá:

1. `Age` y `Flight Distance` no son monótonas en log-odds (meseta entre 40 y 60 años, salto por encima
   de ~1.200 millas), así que discretizarlas debería ayudar.
2. El 0 de la encuesta es una categoría aparte en los ítems digitales, así que tratar los ítems como
   categóricos (o marcar los ceros) debería ayudar.
3. Las demoras sólo pesan como umbral (más de 5 minutos).
4. El efecto del tipo de viaje cambia según `Class`, `Customer Type` y la nota de wifi. Un logit
   aditivo no puede representarlo sin cruces explícitos.

**Dataset:** Playground Series S6E10

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["figure.dpi"] = 100

SEED = 42

## 1. Carga y Preprocesamiento

In [2]:
train_raw = pd.read_csv("../data/train.csv")
test_raw  = pd.read_csv("../data/test.csv")

TARGET = "satisfaction"

crudas_num = ["Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]
encuesta = ["Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
            "Gate location", "Food and drink", "Online boarding", "Seat comfort",
            "Inflight entertainment", "On-board service", "Leg room service", "Baggage handling",
            "Checkin service", "Cleanliness"]
crudas_cat = ["Gender", "Customer Type", "Type of Travel", "Class"]
digitales = ["Inflight wifi service", "Ease of Online booking", "Online boarding"]

# Cortes de distancia por deciles de train, aplicados igual a train y test (no usan el target)
CORTES_DIST = np.unique(np.quantile(train_raw["Flight Distance"], np.linspace(0, 1, 11)))
CORTES_DIST[0], CORTES_DIST[-1] = -np.inf, np.inf


def agregar_derivadas(df):
    """Replica las features derivadas motivadas en 1_eda.ipynb."""
    d = df.copy()
    # Unico hueco del dataset: la demora de llegada se reconstruye con la de salida.
    d["Arrival Delay in Minutes"] = d["Arrival Delay in Minutes"].fillna(d["Departure Delay in Minutes"])
    # Formas no monotonas: tramos de edad de 5 anios y deciles de distancia como categoricas.
    d["age_bin"]  = pd.cut(d["Age"], range(0, 95, 5)).astype(str)
    d["dist_bin"] = pd.cut(d["Flight Distance"], CORTES_DIST).astype(str)
    d["log_dist"] = np.log1p(d["Flight Distance"])
    # Las demoras pesan como umbral, no por su magnitud.
    d["dep_delay_5"] = (d["Departure Delay in Minutes"] > 5).astype(int)
    d["arr_delay_5"] = (d["Arrival Delay in Minutes"] > 5).astype(int)
    # El 0 de los items digitales es "no aplica", no la peor nota.
    for c in digitales:
        d[f"{c}_cero"] = (d[c] == 0).astype(int)
    d["n_ceros"] = (d[encuesta] == 0).sum(axis=1)
    # Encuesta como categoricas: cada nota con su propio efecto.
    for c in encuesta:
        d[f"{c}_cat"] = d[c].astype(str)
    # Cruces con el tipo de viaje.
    d["class_x_travel"] = d["Class"] + "_" + d["Type of Travel"]
    d["cust_x_travel"]  = d["Customer Type"] + "_" + d["Type of Travel"]
    d["wifi_x_travel"]  = d["Inflight wifi service"].astype(str) + "_" + d["Type of Travel"]
    return d


derivadas_num = ["log_dist", "dep_delay_5", "arr_delay_5", "n_ceros"] + [f"{c}_cero" for c in digitales]
derivadas_cat = ["age_bin", "dist_bin"]
encuesta_cat  = [f"{c}_cat" for c in encuesta]
cruces        = ["class_x_travel", "cust_x_travel", "wifi_x_travel"]

train = agregar_derivadas(train_raw)
test  = agregar_derivadas(test_raw)
y     = train[TARGET].astype(int)

print(f"Train: {train_raw.shape[0]:,} filas x {train_raw.shape[1]} columnas")
print(f"Test:  {test_raw.shape[0]:,} filas x {test_raw.shape[1]} columnas")
print(f"Tasa base: {y.mean():.4%}")
print(f"Nulos restantes en Arrival Delay: train {train['Arrival Delay in Minutes'].isna().sum()}"
      f" | test {test['Arrival Delay in Minutes'].isna().sum()}")
print(f"\nDerivadas numericas   ({len(derivadas_num)}): {derivadas_num}")
print(f"Derivadas categoricas ({len(derivadas_cat)}): {derivadas_cat}")
print(f"Cruces                ({len(cruces)}): {cruces}")

Train: 699,635 filas x 23 columnas
Test:  299,844 filas x 22 columnas
Tasa base: 44.3573%
Nulos restantes en Arrival Delay: train 0 | test 0

Derivadas numericas   (7): ['log_dist', 'dep_delay_5', 'arr_delay_5', 'n_ceros', 'Inflight wifi service_cero', 'Ease of Online booking_cero', 'Online boarding_cero']
Derivadas categoricas (2): ['age_bin', 'dist_bin']
Cruces                (3): ['class_x_travel', 'cust_x_travel', 'wifi_x_travel']


## 2. Pipeline de Preprocesamiento

El `Pipeline` escala las numéricas y hace one-hot de las categóricas **dentro** de la validación
cruzada, así que medias y niveles se calculan sólo con el fold de entrenamiento.

Se comparan cinco conjuntos de features para aislar cuánto aporta cada bloque:

- **A**: `Online boarding` sola (la mejor variable del EDA, AUC univariado 0,84). Es el piso.
- **B**: todas las crudas, con la encuesta como numérica.
- **C**: B + derivadas (tramos de edad y distancia, umbrales de demora, marcas de cero). Pone a prueba
  las hipótesis 1-3.
- **D**: como C pero con la encuesta como categórica (cada nota con su propio coeficiente). Es la
  versión fuerte de la hipótesis 2.
- **E**: D + cruces con el tipo de viaje. Pone a prueba la hipótesis 4.

In [3]:
def construir_pipeline(num_cols, cat_cols, C=1.0):
    bloques = [("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                 ("esc", StandardScaler())]), num_cols)]
    if cat_cols:
        bloques.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                         ("ohe", OneHotEncoder(handle_unknown="ignore",
                                                               drop="first"))]), cat_cols))
    return Pipeline([("pre", ColumnTransformer(bloques)),
                     ("clf", LogisticRegression(C=C, max_iter=2000, random_state=SEED))])


configs = {
    "A. solo Online boarding":   (["Online boarding"], []),
    "B. crudas":                 (crudas_num + encuesta, crudas_cat),
    "C. crudas + derivadas":     (crudas_num + encuesta + derivadas_num, crudas_cat + derivadas_cat),
    "D. C con encuesta cat.":    (crudas_num + derivadas_num, crudas_cat + derivadas_cat + encuesta_cat),
    "E. D + cruces de viaje":    (crudas_num + derivadas_num,
                                  crudas_cat + derivadas_cat + encuesta_cat + cruces),
}

for nombre, (n, c) in configs.items():
    print(f"{nombre:28s} -> {len(n)} numericas + {len(c)} categoricas")

A. solo Online boarding      -> 1 numericas + 0 categoricas
B. crudas                    -> 17 numericas + 4 categoricas
C. crudas + derivadas        -> 24 numericas + 6 categoricas
D. C con encuesta cat.       -> 11 numericas + 19 categoricas
E. D + cruces de viaje       -> 11 numericas + 22 categoricas


## 3. Validación Cross-Validation (ROC AUC)

In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

resultados = {}
print(f"{'='*66}")
print("CROSS-VALIDATION 5-FOLD - ROC AUC")
print(f"{'='*66}")
for nombre, (num_cols, cat_cols) in configs.items():
    scores = cross_val_score(construir_pipeline(num_cols, cat_cols),
                             train[num_cols + cat_cols], y, cv=skf,
                             scoring="roc_auc", n_jobs=4)
    resultados[nombre] = scores
    print(f"  {nombre:28s} AUC = {scores.mean():.5f} +/- {scores.std():.5f}")
print(f"{'='*66}")

MEJOR = max(resultados, key=lambda k: resultados[k].mean())
print(f"  Mejor configuracion: {MEJOR}")
print(f"  CV AUC:              {resultados[MEJOR].mean():.5f}")

CROSS-VALIDATION 5-FOLD - ROC AUC
  A. solo Online boarding      AUC = 0.84040 +/- 0.00112
  B. crudas                    AUC = 0.92155 +/- 0.00043
  C. crudas + derivadas        AUC = 0.94110 +/- 0.00050
  D. C con encuesta cat.       AUC = 0.94891 +/- 0.00050
  E. D + cruces de viaje       AUC = 0.94909 +/- 0.00051
  Mejor configuracion: E. D + cruces de viaje
  CV AUC:              0.94909


In [5]:
# Cuanto aporta cada bloque respecto del anterior
base = None
print("Aporte incremental:")
for nombre in configs:
    m = resultados[nombre].mean()
    delta = "" if base is None else f"   ({m - base:+.5f} vs anterior)"
    print(f"  {nombre:28s} {m:.5f}{delta}")
    base = m

Aporte incremental:
  A. solo Online boarding      0.84040
  B. crudas                    0.92155   (+0.08116 vs anterior)
  C. crudas + derivadas        0.94110   (+0.01955 vs anterior)
  D. C con encuesta cat.       0.94891   (+0.00781 vs anterior)
  E. D + cruces de viaje       0.94909   (+0.00018 vs anterior)


## 4. Regularización

Con decenas de columnas one-hot conviene revisar si la regularización por defecto (`C=1`) es la
adecuada. Se barre `C` sobre la mejor configuración.

In [ ]:
num_cols, cat_cols = configs[MEJOR]
features = num_cols + cat_cols

por_C = {}
for C in [0.01, 0.1, 1.0, 10.0]:
    s = cross_val_score(construir_pipeline(num_cols, cat_cols, C=C), train[features], y,
                        cv=skf, scoring="roc_auc", n_jobs=4)
    por_C[C] = s.mean()
    print(f"  C = {C:<6} AUC = {s.mean():.5f} +/- {s.std():.5f}")

MEJOR_C = max(por_C, key=por_C.get)
cv_mean = por_C[MEJOR_C]
print(f"\nMejor C: {MEJOR_C}  ->  CV AUC {cv_mean:.5f}")

## 5. Diagnóstico — Curva ROC y Separación de Probabilidades

In [ ]:
oof = cross_val_predict(construir_pipeline(num_cols, cat_cols, C=MEJOR_C), train[features], y,
                        cv=skf, method="predict_proba", n_jobs=4)[:, 1]
auc_oof = roc_auc_score(y, oof)
print(f"AUC out-of-fold: {auc_oof:.5f}   (media de CV: {cv_mean:.5f})")

fpr, tpr, _ = roc_curve(y, oof)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].plot(fpr, tpr, color="tomato", lw=2, label=f"logit (AUC = {auc_oof:.4f})")
axes[0].plot([0, 1], [0, 1], color="gray", ls=":", lw=1.5, label="azar")
axes[0].set_xlabel("Tasa de falsos positivos")
axes[0].set_ylabel("Tasa de verdaderos positivos")
axes[0].set_title("Curva ROC (predicciones out-of-fold)")
axes[0].legend(loc="lower right")
for cls, color, etiqueta in [(0, "steelblue", "False"), (1, "tomato", "True")]:
    axes[1].hist(oof[y == cls], bins=60, alpha=0.6, color=color,
                 label=f"satisfaction = {etiqueta}")
axes[1].set_xlabel("Probabilidad predicha")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Separacion de las probabilidades out-of-fold")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Donde se equivoca: AUC out-of-fold dentro de cada segmento grande
seg = pd.DataFrame({"y": y, "p": oof, "travel": train["Type of Travel"], "clase": train["Class"]})
print(seg.groupby(["travel", "clase"]).apply(
    lambda g: pd.Series({"n": len(g), "tasa": g["y"].mean(),
                         "auc": roc_auc_score(g["y"], g["p"]) if g["y"].nunique() > 1 else np.nan})
).round(4).to_string())

## 6. Modelo Final — Entrenamiento sobre todo el Train

In [ ]:
modelo_final = construir_pipeline(num_cols, cat_cols, C=MEJOR_C)
modelo_final.fit(train[features], y)
print(f"Modelo final entrenado sobre {len(train):,} filas y {len(features)} features de entrada "
      f"({len(modelo_final.named_steps['pre'].get_feature_names_out())} columnas tras el one-hot).")

## 7. Coeficientes del Modelo

Las numéricas están estandarizadas, así que su coeficiente es el cambio en log-odds ante **un desvío
estándar**. Los de las categóricas se leen contra el nivel de referencia que descarta el one-hot.

In [ ]:
nombres = modelo_final.named_steps["pre"].get_feature_names_out()
coefs   = pd.Series(modelo_final.named_steps["clf"].coef_[0], index=nombres)
top     = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(25).sort_values()

fig, ax = plt.subplots(figsize=(11, 10))
top.plot.barh(ax=ax, color=["steelblue" if v < 0 else "tomato" for v in top], edgecolor="black")
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Coeficiente (log-odds)")
ax.set_title("Top 25 coeficientes de la regresion logistica", fontsize=14)
plt.tight_layout()
plt.show()

print("Top 15 por magnitud:")
print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(15).round(4).to_string())

## 8. Predicción sobre Test y Submission

In [ ]:
proba_test = modelo_final.predict_proba(test[features])[:, 1]

# Se guardan OOF y test para poder stackear este modelo mas adelante
np.save("../models/2_logit_oof.npy", oof)
np.save("../models/2_logit_test.npy", proba_test)

submission = pd.DataFrame({"id": test["id"], TARGET: proba_test})
submission.to_csv("../submissions/2_logit_submission.csv", index=False)

print(f"Submission escrita: ../submissions/2_logit_submission.csv ({len(submission):,} filas)")
print(submission.head().to_string(index=False))
print(f"\nProbabilidad media predicha: {proba_test.mean():.4f}"
      f"  (tasa base en train: {y.mean():.4f})")

In [ ]:
# Chequeo de sanidad contra sample_submission (formato e ids)
sample = pd.read_csv("../data/sample_submission.csv")
assert list(submission.columns) == list(sample.columns), "columnas no coinciden"
assert len(submission) == len(sample), "cantidad de filas no coincide"
assert (submission["id"].values == sample["id"].values).all(), "los ids no estan alineados"
assert submission[TARGET].between(0, 1).all(), "hay probabilidades fuera de [0, 1]"
assert submission[TARGET].notna().all(), "hay nulos en la prediccion"
print("Chequeos OK - submission lista para enviar.")

## 9. Envío a Kaggle vía API

In [ ]:
COMPETITION = "playground-series-s6e10"
FILE        = "../submissions/2_logit_submission.csv"
MSG         = f"Logit - {MEJOR} - C={MEJOR_C} - CV AUC={cv_mean:.5f}"

!kaggle competitions submit -c {COMPETITION} -f "{FILE}" -m "{MSG}"

## 10. Conclusiones

_(se completan después de ejecutar)_